# Checkpoint 7 — Apply the outcome maturity policy

**Goal:** apply our provisional 48-hour reporting allowance and see which examples could train a model at a historical cutoff. This notebook runs independently with `.venv`.

For a decision at Monday 11 AM, the outcome window ends Monday 5 PM. With a 48-hour allowance, the example becomes eligible Wednesday 5 PM. This rule applies to potential positives and negatives alike.

**Eligibility time = decision time + 6 hours + reporting allowance.**

At the training cutoff, use only reports already available. An eligible row with a matching known incident gets label 1. An eligible row without one gets label 0 **under our complete-records assumption**. An immature row gets no label and stays out of training.

No model is trained here. The 48-hour assumption is not proof of completeness or a reporting SLA.


In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "data/labels.jsonl").is_file()
             and (p / "src/dispatch_risk/contracts.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run from inside the candidate repository.")

def load(name):
    with (ROOT / "data" / name).open() as handle:
        return [json.loads(line) for line in handle if line.strip()]

decisions = pd.DataFrame(load("decision_times.jsonl"))
reports = pd.DataFrame(load("labels.jsonl"))
decisions["decision_time"] = pd.to_datetime(decisions["decision_time"], utc=True)
for field in ("incident_at", "label_available_at"):
    reports[field] = pd.to_datetime(reports[field], utc=True)
HORIZON = pd.Timedelta(hours=6)
print("Loaded", len(decisions), "checkpoints and", len(reports), "audited incidents.")


Loaded 1800 checkpoints and 133 audited incidents.


## 1. Choose a demonstration cutoff without looking at incident outcomes

We choose the timestamp at approximately 70% of the sorted unique decision timestamps. This provides a historical point with earlier and later checkpoints. It is a **demonstration cutoff**, not our final train/test split, and not a guarantee of complete observation. No incident labels select it.

Rows at or after that cutoff are outside the historical training period. Rows before it may still be too recent to meet the maturity rule.


In [2]:
unique_times = sorted(decisions["decision_time"].unique())
if not unique_times:
    raise ValueError("No decision timestamps to inspect")
cutoff = pd.Timestamp(unique_times[min(int(len(unique_times) * 0.70), len(unique_times) - 1)])
print("Demonstration training cutoff:", cutoff.isoformat())


Demonstration training cutoff: 2026-02-22T23:00:00+00:00


In [3]:
def eligibility_table(checkpoints, incident_reports, training_cutoff, grace_hours=48):
    if training_cutoff.tzinfo is None:
        raise ValueError("Training cutoff must be timezone-aware")
    if grace_hours < 0:
        raise ValueError("Reporting allowance cannot be negative")
    grace = pd.Timedelta(hours=grace_hours)
    # Future report contents do not participate in historical label construction.
    known = incident_reports.loc[incident_reports["label_available_at"] <= training_cutoff]
    grouped = {sid: group for sid, group in known.groupby("shipment_id", sort=False)}
    result = []
    for row in checkpoints.itertuples(index=False):
        end = row.decision_time + HORIZON
        eligible_at = end + grace
        label = None
        if row.decision_time >= training_cutoff:
            status = "outside_training_period"
        elif eligible_at > training_cutoff:
            status = "waiting_for_maturity"
        else:
            group = grouped.get(row.shipment_id)
            positive = group is not None and bool(((group["incident_at"] > row.decision_time)
                                                   & (group["incident_at"] <= end)).any())
            label = int(positive)
            status = "eligible_positive" if positive else "eligible_assumed_negative"
        result.append({"shipment_id": row.shipment_id, "decision_time": row.decision_time,
                       "horizon_end": end, "eligible_at": eligible_at,
                       "training_cutoff": training_cutoff, "grace_hours": grace_hours,
                       "status": status, "label": label})
    output = pd.DataFrame(result)
    output["label"] = output["label"].astype("Int64")
    return output

maturity = eligibility_table(decisions, reports, cutoff)
display(maturity.groupby("status", sort=True).size().rename("rows").to_frame())
display(maturity.sort_values(["eligible_at", "shipment_id"]).loc[
    lambda t: (t["eligible_at"] - cutoff).abs() <= pd.Timedelta(hours=12)
].head(12))


,rows
status,
eligible_assumed_negative,1113
eligible_positive,96
outside_training_period,540
waiting_for_maturity,51


,shipment_id,decision_time,horizon_end,eligible_at,training_cutoff,grace_hours,status,label
1193,s-00397,2026-02-20 05:00:00+00:00,2026-02-20 11:00:00+00:00,2026-02-22 11:00:00+00:00,2026-02-22 23:00:00+00:00,48,eligible_assumed_negative,0
1195,s-00398,2026-02-20 05:00:00+00:00,2026-02-20 11:00:00+00:00,2026-02-22 11:00:00+00:00,2026-02-22 23:00:00+00:00,48,eligible_assumed_negative,0
1197,s-00399,2026-02-20 05:00:00+00:00,2026-02-20 11:00:00+00:00,2026-02-22 11:00:00+00:00,2026-02-22 23:00:00+00:00,48,eligible_assumed_negative,0
1196,s-00398,2026-02-20 08:00:00+00:00,2026-02-20 14:00:00+00:00,2026-02-22 14:00:00+00:00,2026-02-22 23:00:00+00:00,48,eligible_positive,1
1198,s-00399,2026-02-20 08:00:00+00:00,2026-02-20 14:00:00+00:00,2026-02-22 14:00:00+00:00,2026-02-22 23:00:00+00:00,48,eligible_assumed_negative,0
1200,s-00400,2026-02-20 08:00:00+00:00,2026-02-20 14:00:00+00:00,2026-02-22 14:00:00+00:00,2026-02-22 23:00:00+00:00,48,eligible_assumed_negative,0
1199,s-00399,2026-02-20 11:00:00+00:00,2026-02-20 17:00:00+00:00,2026-02-22 17:00:00+00:00,2026-02-22 23:00:00+00:00,48,eligible_assumed_negative,0
1201,s-00400,2026-02-20 11:00:00+00:00,2026-02-20 17:00:00+00:00,2026-02-22 17:00:00+00:00,2026-02-22 23:00:00+00:00,48,eligible_assumed_negative,0
1203,s-00401,2026-02-20 11:00:00+00:00,2026-02-20 17:00:00+00:00,2026-02-22 17:00:00+00:00,2026-02-22 23:00:00+00:00,48,eligible_assumed_negative,0
1202,s-00400,2026-02-20 14:00:00+00:00,2026-02-20 20:00:00+00:00,2026-02-22 20:00:00+00:00,2026-02-22 23:00:00+00:00,48,eligible_assumed_negative,0


## 2. Verify the policy with a small boundary example

An incident occurs Monday afternoon but its report is not available until Thursday. At Wednesday's maturity time, our 48-hour assumption would assign a negative. On Thursday, the newly known report contradicts it. This deliberately shows that the allowance is an assumption, not a guarantee.

Before maturity, even a positive already reported remains excluded under our symmetric eligibility policy.


In [4]:
example = pd.DataFrame({"shipment_id": ["teaching-shipment"],
                        "decision_time": [pd.Timestamp("2026-01-05T11:00:00Z")]})
late_report = pd.DataFrame({"shipment_id": ["teaching-shipment"],
                           "incident_at": [pd.Timestamp("2026-01-05T15:00:00Z")],
                           "label_available_at": [pd.Timestamp("2026-01-08T09:00:00Z")]})
ready = pd.Timestamp("2026-01-07T17:00:00Z")
before = eligibility_table(example, late_report, ready - pd.Timedelta(seconds=1))
at = eligibility_table(example, late_report, ready)
later = eligibility_table(example, late_report, pd.Timestamp("2026-01-08T10:00:00Z"))
assert pd.isna(before.iloc[0]["label"])
assert at.iloc[0]["label"] == 0
assert later.iloc[0]["label"] == 1
assert eligibility_table(example, late_report.iloc[:0], ready).equals(at)
quick_report = late_report.copy()
quick_report["label_available_at"] = pd.Timestamp("2026-01-06T09:00:00Z")
assert pd.isna(eligibility_table(example, quick_report, ready - pd.Timedelta(seconds=1)).iloc[0]["label"])
assert eligibility_table(example, quick_report, ready).iloc[0]["label"] == 1
assert maturity.loc[maturity["label"].notna(), "eligible_at"].le(cutoff).all()
assert maturity.loc[maturity["status"] == "outside_training_period", "label"].isna().all()
display(pd.concat([before, at, later], ignore_index=True))
print("Passed: maturity boundary, future-report exclusion, symmetric waiting, and late-report contradiction example.")


,shipment_id,decision_time,horizon_end,eligible_at,training_cutoff,grace_hours,status,label
0,teaching-shipment,2026-01-05 11:00:00+00:00,2026-01-05 17:00:00+00:00,2026-01-07 17:00:00+00:00,2026-01-07 16:59:59+00:00,48,waiting_for_maturity,<NA>
1,teaching-shipment,2026-01-05 11:00:00+00:00,2026-01-05 17:00:00+00:00,2026-01-07 17:00:00+00:00,2026-01-07 17:00:00+00:00,48,eligible_assumed_negative,0
2,teaching-shipment,2026-01-05 11:00:00+00:00,2026-01-05 17:00:00+00:00,2026-01-07 17:00:00+00:00,2026-01-08 10:00:00+00:00,48,eligible_positive,1


Passed: maturity boundary, future-report exclusion, symmetric waiting, and late-report contradiction example.


## 3. Compare reporting allowances at the same cutoff

Compare 48, 72, and 96 hours. Longer allowances can exclude more recent rows. These counts are an **eligibility comparison**, not a model performance result. We cannot choose a winning allowance from accuracy because no model has been evaluated.

The retrospective contradiction count below uses reports arriving after the cutoff strictly for auditing. It does not alter historical training labels. Zero contradictions in the supplied table would not prove every real-world incident is reported.


In [5]:
comparisons = []
for hours in (48, 72, 96):
    table = eligibility_table(decisions, reports, cutoff, hours)
    negative_rows = table.loc[table["label"].eq(0).fillna(False)]
    contradictions = 0
    for row in negative_rows.itertuples(index=False):
        match = reports.loc[(reports["shipment_id"] == row.shipment_id)
                            & (reports["incident_at"] > row.decision_time)
                            & (reports["incident_at"] <= row.horizon_end)
                            & (reports["label_available_at"] > cutoff)]
        contradictions += int(not match.empty)
    comparisons.append({"grace_hours": hours,
                        "eligible_rows": int(table["label"].notna().sum()),
                        "positives": int(table["label"].eq(1).sum()),
                        "assumed_negatives": int(table["label"].eq(0).sum()),
                        "waiting_rows": int(table["status"].eq("waiting_for_maturity").sum()),
                        "outside_training_period": int(table["status"].eq("outside_training_period").sum()),
                        "later_report_contradictions": contradictions})
comparison = pd.DataFrame(comparisons)
display(comparison)
assert comparison["eligible_rows"].is_monotonic_decreasing
print(comparison.to_string(index=False))


,grace_hours,eligible_rows,positives,assumed_negatives,waiting_rows,outside_training_period,later_report_contradictions
0,48,1209,96,1113,51,540,0
1,72,1185,94,1091,75,540,0
2,96,1161,93,1068,99,540,0


 grace_hours  eligible_rows  positives  assumed_negatives  waiting_rows  outside_training_period  later_report_contradictions
          48           1209         96               1113            51                      540                            0
          72           1185         94               1091            75                      540                            0
          96           1161         93               1068            99                      540                            0


## 4. What we have and what we do not yet have

We have an explicit experimental label policy, a traceable eligibility table, and a reporting-allowance comparison. No features have been joined to these labels, no model has been fitted, and no final evaluation split has been selected.

The synthetic inputs support our experiment, not proof of production reporting coverage. Final design must address training-cutoff metadata through the fixed public interface, a defensible observation end for held-out outcomes, possible same-shipment overlap, and overlapping outcome horizons.

**Interview notes:** “I applied the same outcome-maturity rule to both classes and restricted training labels to reports available by the historical fit cutoff. I also audited whether later reports contradicted assumed negatives.”

**Try explaining this:** if a checkpoint is too recent, should we mark it zero or keep it out of training for now?

**Next checkpoint:** combine reviewed features with eligible labels to inspect a first training table, then learn baseline/model fitting with a properly defined temporal evaluation design. Results here do not establish model quality.
